# Baseline: TF-IDF + Logistic Regression
**Dataset:** IMDB 50K Movie Reviews (HuggingFace)  
**Goal:** Establish a strong classical ML baseline for sentiment classification.

## 1. Imports & Setup

In [ ]:
import re
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Setup complete.')

## 2. Load Dataset

In [ ]:
# Load IMDB dataset from HuggingFace (25k train / 25k test)
dataset = load_dataset('imdb')

train_texts = dataset['train']['text']
train_labels = dataset['train']['label']
test_texts  = dataset['test']['text']
test_labels = dataset['test']['label']

print(f'Train samples : {len(train_texts):,}')
print(f'Test  samples : {len(test_texts):,}')
print(f'Label mapping : 0 = Negative, 1 = Positive')
print(f'\nSample review:\n{train_texts[0][:300]}...')

## 3. Text Preprocessing

In [ ]:
def preprocess(text: str) -> str:
    """Clean raw review text for TF-IDF ingestion."""
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)          # strip HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)         # remove punctuation / digits
    text = re.sub(r'\s+', ' ', text).strip()       # collapse whitespace
    return text

train_clean = [preprocess(t) for t in train_texts]
test_clean  = [preprocess(t) for t in test_texts]

# Verify preprocessing
print('Raw    :', train_texts[0][:120])
print('Cleaned:', train_clean[0][:120])

## 4. TF-IDF Vectorization

In [ ]:
# Unigrams + bigrams, cap vocabulary at 50k most frequent terms
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,       # apply 1 + log(tf) scaling
    min_df=2,                # ignore very rare terms
    strip_accents='unicode'
)

X_train = tfidf.fit_transform(train_clean)
X_test  = tfidf.transform(test_clean)

print(f'Train matrix shape : {X_train.shape}')
print(f'Test  matrix shape : {X_test.shape}')
print(f'Sparsity           : {1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.4%}')

## 5. Train Logistic Regression

In [ ]:
clf = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    random_state=RANDOM_SEED,
    n_jobs=-1
)

t0 = time.time()
clf.fit(X_train, train_labels)
train_time = time.time() - t0

print(f'Training time: {train_time:.2f}s')

## 6. Evaluation

In [ ]:
# Predict and measure inference latency
t0 = time.time()
preds = clf.predict(X_test)
latency_ms = (time.time() - t0) / len(test_labels) * 1000

acc = accuracy_score(test_labels, preds)
f1  = f1_score(test_labels, preds, average='binary')

print(f'Accuracy         : {acc:.4f}')
print(f'F1 Score         : {f1:.4f}')
print(f'Latency/sample   : {latency_ms:.4f} ms')
print()
print(classification_report(test_labels, preds, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Baseline (TF-IDF + LR) — Confusion Matrix')
plt.tight_layout()
plt.savefig('results/baseline_confusion_matrix.png', dpi=150)
plt.show()

## 7. Top Predictive Features

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_[0]

top_pos = feature_names[np.argsort(coefs)[-20:][::-1]]
top_neg = feature_names[np.argsort(coefs)[:20]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(top_pos, coefs[np.argsort(coefs)[-20:][::-1]], color='steelblue')
axes[0].set_title('Top Positive Features')
axes[0].invert_yaxis()

axes[1].barh(top_neg, coefs[np.argsort(coefs)[:20]], color='tomato')
axes[1].set_title('Top Negative Features')
axes[1].invert_yaxis()

plt.suptitle('Logistic Regression — Most Predictive Features')
plt.tight_layout()
plt.savefig('results/baseline_top_features.png', dpi=150)
plt.show()

## 8. Save Metrics

In [ ]:
metrics = {
    'model'       : 'TF-IDF + Logistic Regression',
    'accuracy'    : round(acc, 4),
    'f1_score'    : round(f1, 4),
    'latency_ms'  : round(latency_ms, 4),
    'train_time_s': round(train_time, 2)
}

with open('results/baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Metrics saved to results/baseline_metrics.json')
print(json.dumps(metrics, indent=2))